# Binding vs Knowledge Knockout — Base Models

This notebook reproduces the paper's **base-model binding/knowledge knockout results**:
baseline pairing effects |&Delta;S| (binding) and |&Delta;K| (knowledge), the R&rarr;item
edge-knockout reductions with the U&rarr;item control knockout, the K/S dissociation
ratio, and the baseline significance / directional-differentiation statistics
(base rows of the knockout results tables). A final validation gate checks the
regenerated numbers against the published run (`TARGETS_BASE` in
`common/published_targets.py`).

This single notebook covers all 4 base models via `MODEL_KEY` (note the base key is
`"gemma"`, unlike the instruct key `"gemma2"`).

Requires `./data/N4_1k.pkl` and a GPU. Results are saved under
`./results/<model>_base/`.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma", "nemo"}


In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "base")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN            # read from the HF_TOKEN environment variable
MC_WANG = config.MC_WANG
ANSWER_START = config.ANSWER_START    # set by config.init() from CFG["cond_approach"]
MAX_NEW_TOKENS = config.MAX_NEW_TOKENS
HEADS = config.HEADS
OUTPUT_DIR = config.OUTPUT_DIR        # ./results/<model>_base/

# Imports
import numpy as np
import os
import re
import ast
import gc
import pickle
import torch
import torch.nn.functional as F
import importlib
import unicodedata

from pathlib import Path
from collections import Counter, defaultdict
from contextlib import contextmanager
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM

from common.text_parsers import (norm_identity, extract_options, extract_scenario,
                                 parse_meta, replace_options)
from common.base.data import (_display_form, extract_neutral_item, load_n4,
                              build_factorial_as_conditions, build_knowledge_probes,
                              KNOWLEDGE_OPT_C)
from common.base.prompts import (find_option_token_ids, format_for_base,
                                 discover_after_paren_ids,
                                 discover_after_paren_ids_in_prompt,
                                 find_token_spans, extract_options_text,
                                 find_identity_and_item_positions)
from common.base.hooks import edge_knockout, edge_scale
from common.base.scoring import _compute_scores
from common.base.store import save_results
from common.stats_utils import cond_p_R

print(f"Selected model: {MODEL_KEY}")
print(f"  Path:            {CFG['path']}")
print(f"  Binding heads:   {CFG['heads']}")
print(f"  Soft-capping:    {CFG['softcap']}")
print(f"  Cond. approach:  {CFG['cond_approach']} (token: {repr(CFG['cond_token_str'])})")
print(f"ANSWER_START ends with: ...{repr(ANSWER_START[-10:])}")


In [ ]:
"""
Factorial condition builder for N4 cultural binding pairs.

Builds the (B_cult, B_unrel) condition pairs used throughout the pipeline.
Identity strings preserve their original capitalisation from the dataset
(e.g. "Turkish person"); a normalised form is used internally for matching.
"""

# Helper functions (norm_identity, extract_options, extract_scenario, parse_meta,
# replace_options, _display_form, extract_neutral_item, load_n4,
# build_factorial_pairs_v3, build_factorial_as_conditions) are imported from
# common/ (see setup cell). build_6_conditions is kept in this cell: it is not
# used by this pipeline and was therefore not extracted to common/.

# ================================================================
# build_6_conditions v3 (CASE-PRESERVING)
# ================================================================

MAX_RETRY = 5

def build_6_conditions(cultural_items, neutral_items, seed=42):
    """
    Build 8 conditions for gradient + pairing analysis.

    v3 change: preserves original capitalisation when substituting
    identities into prompts.  norm_identity() is used only for
    matching / deduplication, never for prompt text.
    """
    rng = np.random.RandomState(seed)

    # Build pools: track BOTH normalised key and display form
    all_cultural_ids = set()
    all_noncultural_ids = set()
    norm_to_display = {}          # norm → cased form (no article)
    item_data_cult = []
    group_to_items = defaultdict(set)

    for item in cultural_items:
        q, ans, meta = item
        try:
            qtype, c_item, in_groups, out_groups = parse_meta(meta)
        except Exception:
            item_data_cult.append(None)
            continue
        oa, ob = extract_options(q)
        if oa is None:
            item_data_cult.append(None)
            continue

        for g in in_groups:
            gn = norm_identity(g)
            all_cultural_ids.add(gn)
            group_to_items[gn].add(c_item)
            # Keep the FIRST cased form we encounter
            if gn not in norm_to_display:
                norm_to_display[gn] = _display_form(g)

        for g in out_groups:
            gn = norm_identity(g)
            all_noncultural_ids.add(gn)
            if gn not in norm_to_display:
                norm_to_display[gn] = _display_form(g)

        cult_opt = None
        matched_group = None
        for g in in_groups:
            if norm_identity(g) == norm_identity(oa):
                cult_opt = 'a'
                matched_group = norm_identity(g)
                break
            elif norm_identity(g) == norm_identity(ob):
                cult_opt = 'b'
                matched_group = norm_identity(g)
                break

        item_data_cult.append({
            'q': q, 'ans': ans, 'meta': meta,
            'c_item': c_item, 'in_groups': in_groups, 'out_groups': out_groups,
            'oa': oa, 'ob': ob, 'cult_opt': cult_opt,
            'matched_group': matched_group,
            'cult_text': oa if cult_opt == 'a' else ob if cult_opt else None,
            'noncult_text': ob if cult_opt == 'a' else oa if cult_opt else None,
        })

    # Parse neutral items
    neut_by_scenario = {}
    for item in neutral_items:
        q, ans, meta = item
        scenario = extract_scenario(meta)
        if scenario not in neut_by_scenario:
            neut_by_scenario[scenario] = []
        neut_by_scenario[scenario].append(item)

    noncult_list = sorted(all_noncultural_ids - all_cultural_ids)
    cult_list = sorted(all_cultural_ids)

    # Helper: get the display form for a normalised identity
    def _disp(norm_key):
        """Return cased display string (with 'the' prefix)."""
        return f"the {norm_to_display.get(norm_key, norm_key)}"

    results = {
        'A_cult': [], 'B_cult': [], 'B_unrel': [], 'C_cult': [],
        'A_neut': [], 'B_neut': [], 'B_neut_unrel': [], 'C_neut': [],
        'scenarios': [], 'items_cult': [], 'items_neut': [], 'assoc_pos': [],
        'a_answers': [],
    }

    valid_count = 0
    skipped = defaultdict(int)

    for i, d in enumerate(item_data_cult):
        if d is None or d['cult_opt'] is None:
            continue

        q_cult = d['q']
        cult_text = d['cult_text']
        noncult_text = d['noncult_text']
        c_item = d['c_item']
        matched_group = d['matched_group']
        scenario = extract_scenario(d['meta'])
        cult_pos = d['cult_opt']

        # Find paired neutral item
        if scenario not in neut_by_scenario:
            skipped['no_neutral'] += 1
            continue
        neut_candidates = neut_by_scenario[scenario]
        neut_item = neut_candidates[rng.randint(len(neut_candidates))]
        q_neut = neut_item[0]

        oa_n, ob_n = extract_options(q_neut)
        if oa_n is None:
            skipped['neut_no_options'] += 1
            continue

        # B_cult: keep associated, replace non-cultural with random cultural
        other_cult = [c for c in cult_list if c != matched_group]
        if not other_cult:
            skipped['no_other_cult'] += 1
            continue

        b_cult_q = None
        swap_cult_b = None
        for attempt in range(MAX_RETRY):
            swap_cult_b = rng.choice(other_cult)
            if cult_pos == 'a':
                candidate = replace_options(q_cult, new_a=_disp(matched_group), new_b=_disp(swap_cult_b))
            else:
                candidate = replace_options(q_cult, new_a=_disp(swap_cult_b), new_b=_disp(matched_group))
            oa_t, ob_t = extract_options(candidate)
            if oa_t and ob_t and norm_identity(oa_t) != norm_identity(ob_t):
                b_cult_q = candidate
                break
        if b_cult_q is None:
            skipped['degenerate_B_cult'] += 1
            continue

        # B_unrel: keep swap_cult_b, replace associated
        unrelated_cult = [g for g in cult_list
                          if c_item not in group_to_items[g]
                          and g != swap_cult_b]
        if not unrelated_cult:
            skipped['not_enough_unrelated'] += 1
            continue

        b_mis_q = None
        swap_unrel = None
        for attempt in range(MAX_RETRY):
            swap_unrel = rng.choice(unrelated_cult)
            if cult_pos == 'a':
                candidate = replace_options(q_cult,
                                            new_a=_disp(swap_unrel),
                                            new_b=_disp(swap_cult_b))
            else:
                candidate = replace_options(q_cult,
                                            new_a=_disp(swap_cult_b),
                                            new_b=_disp(swap_unrel))
            oa_t, ob_t = extract_options(candidate)
            if oa_t and ob_t and norm_identity(oa_t) != norm_identity(ob_t):
                b_mis_q = candidate
                break
        if b_mis_q is None:
            skipped['degenerate_B_unrel'] += 1
            continue

        # C_cult: replace cultural identity with non-cultural
        c_cult_q = None
        swap_noncult = None
        for attempt in range(MAX_RETRY):
            swap_noncult = rng.choice(noncult_list)
            if cult_pos == 'a':
                candidate = replace_options(q_cult, new_a=_disp(matched_group), new_b=_disp(swap_noncult))
            else:
                candidate = replace_options(q_cult, new_a=_disp(swap_noncult), new_b=_disp(matched_group))
            oa_t, ob_t = extract_options(candidate)
            if oa_t and ob_t and norm_identity(oa_t) != norm_identity(ob_t):
                c_cult_q = candidate
                break
        if c_cult_q is None:
            skipped['degenerate_C_cult'] += 1
            continue

        # Build neutral conditions (same identity pairs, neutral item)
        a_neut_q = replace_options(q_neut, new_a=_disp(norm_identity(d['oa'])), new_b=_disp(norm_identity(d['ob'])))

        b_cult_oa, b_cult_ob = extract_options(b_cult_q)
        b_neut_q = replace_options(q_neut, new_a=b_cult_oa, new_b=b_cult_ob)

        bmis_oa, bmis_ob = extract_options(b_mis_q)
        b_neut_mis_q = replace_options(q_neut, new_a=bmis_oa, new_b=bmis_ob)

        c_cult_oa, c_cult_ob = extract_options(c_cult_q)
        c_neut_q = replace_options(q_neut, new_a=c_cult_oa, new_b=c_cult_ob)

        # Store
        a_cult_q = replace_options(q_cult, new_a=_disp(norm_identity(d['oa'])), new_b=_disp(norm_identity(d['ob'])))
        results['A_cult'].append(a_cult_q)
        results['B_cult'].append(b_cult_q)
        results['B_unrel'].append(b_mis_q)
        results['C_cult'].append(c_cult_q)
        results['A_neut'].append(a_neut_q)
        results['B_neut'].append(b_neut_q)
        results['B_neut_unrel'].append(b_neut_mis_q)
        results['C_neut'].append(c_neut_q)
        results['scenarios'].append(scenario)
        results['items_cult'].append(c_item)
        results['assoc_pos'].append(cult_pos)
        try:
            neut_qtype = int(neut_item[2][:neut_item[2].index("-")])
            neut_item_text = extract_neutral_item(neut_item[0], neut_qtype)
            results['items_neut'].append(neut_item_text)
        except Exception:
            results['items_neut'].append(None)
        results['a_answers'].append(int(d['ans']))
        valid_count += 1

    print(f"  Built {valid_count} valid condition sets")
    if skipped:
        for k, v in sorted(skipped.items()):
            print(f"    Skipped ({k}): {v}")

    # Diagnostic: show first few pairs
    print(f"\n  Sample pairs (v3 — case-preserving, single-variable design):")
    for idx in range(min(5, valid_count)):
        oa_c, ob_c = extract_options(results['B_cult'][idx].split("\n\n")[0])
        oa_u, ob_u = extract_options(results['B_unrel'][idx].split("\n\n")[0])
        pos = results['assoc_pos'][idx]
        item = results['items_cult'][idx]
        print(f"    [{idx}] item={item}")
        print(f"         B_cult:  (a) {oa_c:40s} (b) {ob_c}")
        print(f"         B_unrel: (a) {oa_u:40s} (b) {ob_u}")
        if pos == 'a':
            print(f"         Changed: (a) {oa_c} → {oa_u}  |  (b) kept: {ob_c}")
        else:
            print(f"         Changed: (b) {ob_c} → {ob_u}  |  (a) kept: {oa_c}")

    # Log the display mapping for verification
    print(f"\n  norm_to_display has {len(norm_to_display)} entries. Samples:")
    for k in list(norm_to_display.keys())[:8]:
        print(f"    '{k}' → '{norm_to_display[k]}'")

    return results


cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data["B_cult"])
print(f"  {n_total} factorial pairs, {len(set(data['scenarios']))} scenarios")


In [ ]:
print(f"Loading {CFG['path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG["path"], trust_remote_code=True, token=HF_TOKEN
)
model = AutoModelForCausalLM.from_pretrained(
    CFG["path"], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
first_device = next(model.parameters()).device

# Publish runtime singletons so common/ helpers can reference them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
n_kv = getattr(model.config, "num_key_value_heads", n_heads)
softcap_val = getattr(model.config, "attn_logit_softcapping", None)

print(f"  {n_layers} layers, {n_heads} Q-heads, {n_kv} KV-heads (GQA group={n_heads // n_kv})")
print(f"  Softcapping: {softcap_val}")
print(f"  Search space: {n_layers * n_heads} (layer, head) pairs")

In [ ]:
# find_option_token_ids / format_for_base / discover_after_paren_ids /
# discover_after_paren_ids_in_prompt are imported from common.base.prompts.

# -- Option tokens (bare scoring fallback) --
option_tokens_raw = find_option_token_ids(tokenizer)
option_tokens = {
    opt: torch.tensor(ids, device=first_device)
    for opt, ids in option_tokens_raw.items()
}

# -- Conditional scoring: discover COND_AFTER_IDS from the model --
COND_PREFIX_TID = None
COND_AFTER_IDS = None

_sample = format_for_base(data["B_cult"][:1])[0]

if CFG["cond_approach"] == "prefix":
    COND_PREFIX_TID, COND_AFTER_IDS, _sample_cov = discover_after_paren_ids(
        tokenizer, model, first_device, _sample,
        cond_token_str=CFG["cond_token_str"],
    )
    print(f"Conditional scoring: prefix {repr(CFG['cond_token_str'])} = TID {COND_PREFIX_TID}")
else:
    COND_AFTER_IDS, _sample_cov = discover_after_paren_ids_in_prompt(
        tokenizer, model, first_device, _sample,
    )
    print(f"Conditional scoring: '(' embedded in ANSWER_START (single forward pass)")

print(f"  after_ids: a={COND_AFTER_IDS['a']}, b={COND_AFTER_IDS['b']}, c={COND_AFTER_IDS['c']}")
print(f"  Sample coverage P(a+b+c | '('): {_sample_cov:.4f}")
for opt in ["a", "b", "c"]:
    decoded = [tokenizer.decode([tid]) for tid in COND_AFTER_IDS[opt]]
    print(f"  {opt}: TIDs={COND_AFTER_IDS[opt]} -> {decoded}")

In [ ]:
# find_token_spans / extract_options_text / find_identity_and_item_positions
# are imported from common.base.prompts (see setup cell).
print("Token span detection ready.")


In [ ]:
# edge_knockout / edge_scale are imported from common.base.hooks (see setup cell).
print(f"Edge functions ready (module: {CFG['attn_module'].split('.')[-1]})")


In [ ]:
# ================================================================
# KNOWLEDGE PROBE CONSTRUCTION + RESULTS STORE
# ================================================================
# KNOWLEDGE_OPT_C and build_knowledge_probes are imported from common.base.data.

knowledge = build_knowledge_probes(data)

# Results store
from datetime import datetime
from pathlib import Path

# OUTPUT_DIR is set by config.init() to ./results/<model>_base/ (see setup cell).

def init_results_store():
    return {
        'meta': {
            'model': MODEL_KEY,
            'label': CFG['path'].split('/')[-1],
            'model_path': CFG['path'],
            'variant': 'base',
            'timestamp': datetime.now().isoformat(),
            'seed': SEED,
        },
        'factorial': {
            'n_pairs': n_total,
            'items': list(data['items_cult']),
            'assoc_pos': list(data['assoc_pos']),
            'correct_group': list(data['correct_group']),
            'wrong_group': list(data['wrong_group']),
            'third_group': list(data['third_group']),
        },
        'binding': {}, 'knowledge': {}, 'heads': {},
        'knockout': {}, 'dose_response': {},
    }

# save_results is imported from common.base.store (see setup cell).

store = init_results_store()
print(f"  Store initialized for {MODEL_KEY} (base)")


In [ ]:
# ================================================================
# STANDALONE KNOCKOUT: BINDING + KNOWLEDGE
# ================================================================
# This cell is self-contained. It rebuilds positions and baselines
# from scratch. Only depends on: model, tokenizer, data, knowledge,
# format_for_base, COND_AFTER_IDS, COND_PREFIX_TID, HEADS,
# edge_knockout, find_identity_and_item_positions, store.

print("=" * 70)
print(f"  STANDALONE KO — {CFG['path'].split('/')[-1]}")
print(f"  Heads: {HEADS}")
print("=" * 70)

import time as _time
_t_start = _time.time()

store['heads'] = {str(k): v for k, v in HEADS.items()}

# Format all texts
_conditions = ["B_cult", "B_unrel"]
_k_conditions = ["K_cult", "K_unrel"]
_texts_fmt = {c: format_for_base(data[c]) for c in _conditions}
_texts_fmt_k = {c: format_for_base(knowledge[c]) for c in _k_conditions}


# S-score / K-score function (with optional KO): _compute_scores
#    is imported from common.base.scoring (see setup cell).


# Build positions
print("\n  Building positions...")
_positions = {c: [] for c in _conditions}
for c in _conditions:
    for i in range(n_total):
        pos = find_identity_and_item_positions(
            tokenizer, _texts_fmt[c][i], data[c][i],
            data["items_cult"][i], data["assoc_pos"][i])
        _positions[c].append(pos)
    n_valid = sum(1 for p in _positions[c] if p is not None)
    print(f"    Binding {c}: {n_valid}/{n_total}")

_positions_k = {c: [] for c in _k_conditions}
for c in _k_conditions:
    for i in range(n_total):
        pos = find_identity_and_item_positions(
            tokenizer, _texts_fmt_k[c][i], knowledge[c][i],
            data["items_cult"][i], data["assoc_pos"][i])
        _positions_k[c].append(pos)
    n_valid = sum(1 for p in _positions_k[c] if p is not None)
    print(f"    Knowledge {c}: {n_valid}/{n_total}")


# ================================================================
# A. BINDING
# ================================================================
print(f"\n  [BINDING] Baseline...")
_s_base = {}
_lps_s_base = {}
for c in _conditions:
    _s_base[c], _lps_s_base[c] = _compute_scores(
        model, tokenizer, _texts_fmt[c], None,
        {}, "none", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_s = _s_base["B_cult"].mean() - _s_base["B_unrel"].mean()
_diffs_s_base = _s_base["B_cult"] - _s_base["B_unrel"]
print(f"  Baseline delta(S) = {_delta_s:.4f}")

print(f"\n  [BINDING] B->item KO...")
_s_ko_B = {}
_lps_s_ko_B = {}
for c in _conditions:
    _s_ko_B[c], _lps_s_ko_B[c] = _compute_scores(
        model, tokenizer, _texts_fmt[c], _positions[c],
        HEADS, "B_to_item", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_s_ko = _s_ko_B["B_cult"].mean() - _s_ko_B["B_unrel"].mean()
_diffs_s_ko = _s_ko_B["B_cult"] - _s_ko_B["B_unrel"]
_t_s, _p_s = stats.ttest_rel(_diffs_s_base, _diffs_s_ko)
_s_red = (1 - _delta_s_ko / _delta_s) * 100 if abs(_delta_s) > 1e-10 else 0
print(f"  B->item delta(S) = {_delta_s_ko:.4f}  (reduction: {_s_red:.1f}%)")
print(f"  t = {_t_s:.3f}, p = {_p_s:.6f}")
torch.cuda.empty_cache()

print(f"\n  [BINDING] A->item KO (control)...")
_s_ko_A = {}
_lps_s_ko_A = {}
for c in _conditions:
    _s_ko_A[c], _lps_s_ko_A[c] = _compute_scores(
        model, tokenizer, _texts_fmt[c], _positions[c],
        HEADS, "A_to_item", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_s_ctrl = _s_ko_A["B_cult"].mean() - _s_ko_A["B_unrel"].mean()
_diffs_s_ctrl = _s_ko_A["B_cult"] - _s_ko_A["B_unrel"]
_t_s_A, _p_s_A = stats.ttest_rel(_diffs_s_base, _diffs_s_ctrl)
_s_red_A = (1 - _delta_s_ctrl / _delta_s) * 100 if abs(_delta_s) > 1e-10 else 0
print(f"  A->item delta(S) = {_delta_s_ctrl:.4f}  (reduction: {_s_red_A:.1f}%)")
print(f"  t = {_t_s_A:.3f}, p = {_p_s_A:.6f}")
torch.cuda.empty_cache()

store['knockout'] = {
    'binding': {
        'delta_baseline': float(_delta_s),
        'delta_B_ko': float(_delta_s_ko),
        'delta_A_ko': float(_delta_s_ctrl),
        'reduction_B_pct': float(_s_red),
        'reduction_A_pct': float(_s_red_A),
        't_B': float(_t_s), 'p_B': float(_p_s),
        't_A': float(_t_s_A), 'p_A': float(_p_s_A),
        'diffs_base': _diffs_s_base.tolist(),
        'diffs_B_ko': _diffs_s_ko.tolist(),
        'diffs_A_ko': _diffs_s_ctrl.tolist(),
    }
}


# ================================================================
# B. KNOWLEDGE
# ================================================================
print(f"\n  [KNOWLEDGE] Baseline...")
_k_base = {}
_lps_k_base = {}
for c in _k_conditions:
    _k_base[c], _lps_k_base[c] = _compute_scores(
        model, tokenizer, _texts_fmt_k[c], None,
        {}, "none", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_k = _k_base["K_cult"].mean() - _k_base["K_unrel"].mean()
_diffs_k_base = _k_base["K_cult"] - _k_base["K_unrel"]
print(f"  Baseline delta(K) = {_delta_k:.4f}")

print(f"\n  [KNOWLEDGE] B->item KO...")
_k_ko_B = {}
for c in _k_conditions:
    _k_ko_B[c], _ = _compute_scores(
        model, tokenizer, _texts_fmt_k[c], _positions_k[c],
        HEADS, "B_to_item", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_k_ko = _k_ko_B["K_cult"].mean() - _k_ko_B["K_unrel"].mean()
_diffs_k_ko = _k_ko_B["K_cult"] - _k_ko_B["K_unrel"]
_t_k, _p_k = stats.ttest_rel(_diffs_k_base, _diffs_k_ko)
_k_red = (1 - _delta_k_ko / _delta_k) * 100 if abs(_delta_k) > 1e-10 else 0
print(f"  B->item delta(K) = {_delta_k_ko:.4f}  (reduction: {_k_red:.1f}%)")
print(f"  t = {_t_k:.3f}, p = {_p_k:.6f}")
torch.cuda.empty_cache()

print(f"\n  [KNOWLEDGE] A->item KO (control)...")
_k_ko_A = {}
for c in _k_conditions:
    _k_ko_A[c], _ = _compute_scores(
        model, tokenizer, _texts_fmt_k[c], _positions_k[c],
        HEADS, "A_to_item", COND_AFTER_IDS, COND_PREFIX_TID)
_delta_k_ctrl = _k_ko_A["K_cult"].mean() - _k_ko_A["K_unrel"].mean()
_diffs_k_ctrl = _k_ko_A["K_cult"] - _k_ko_A["K_unrel"]
_t_k_A, _p_k_A = stats.ttest_rel(_diffs_k_base, _diffs_k_ctrl)
_k_red_A = (1 - _delta_k_ctrl / _delta_k) * 100 if abs(_delta_k) > 1e-10 else 0
print(f"  A->item delta(K) = {_delta_k_ctrl:.4f}  (reduction: {_k_red_A:.1f}%)")
print(f"  t = {_t_k_A:.3f}, p = {_p_k_A:.6f}")
torch.cuda.empty_cache()

store['knockout']['knowledge'] = {
    'delta_baseline': float(_delta_k),
    'delta_B_ko': float(_delta_k_ko),
    'delta_A_ko': float(_delta_k_ctrl),
    'reduction_B_pct': float(_k_red),
    'reduction_A_pct': float(_k_red_A),
    't_B': float(_t_k), 'p_B': float(_p_k),
    't_A': float(_t_k_A), 'p_A': float(_p_k_A),
    'diffs_base': _diffs_k_base.tolist(),
    'diffs_B_ko': _diffs_k_ko.tolist(),
    'diffs_A_ko': _diffs_k_ctrl.tolist(),
}


# ================================================================
# DISSOCIATION SUMMARY
# ================================================================
print(f"\n  {'='*55}")
print(f"  DISSOCIATION: BINDING vs KNOWLEDGE under B->item KO")
print(f"  {'='*55}")
print(f"  Binding  |dS| B->item: {_s_red:+.1f}%  (p={_p_s:.4f})")
print(f"  Binding  |dS| A->item: {_s_red_A:+.1f}%  (p={_p_s_A:.4f})  [control]")
print(f"  Knowledge|dK| B->item: {_k_red:+.1f}%  (p={_p_k:.4f})")
print(f"  Knowledge|dK| A->item: {_k_red_A:+.1f}%  (p={_p_k_A:.4f})  [control]")
print(f"  {'='*55}")
if abs(_k_red) < abs(_s_red) / 2:
    print(f"  -> Dissociation: heads mediate GATING, not storage")
elif abs(_k_red) > abs(_s_red):
    print(f"  -> No clear dissociation: heads also affect knowledge")
else:
    print(f"  -> Partial effect on both tasks")


# Significance of baseline pairing effects
from scipy.stats import ttest_1samp

items_arr_sig = np.array(data['items_cult'])
unique_items_sig = np.unique(items_arr_sig)

dS_per_pair = _s_base["B_cult"] - _s_base["B_unrel"]
dK_per_pair = _k_base["K_cult"] - _k_base["K_unrel"]

dS_item = np.array([dS_per_pair[items_arr_sig == it].mean() for it in unique_items_sig])
dK_item = np.array([dK_per_pair[items_arr_sig == it].mean() for it in unique_items_sig])

t_dS, p_dS = ttest_1samp(dS_item, 0)
t_dK, p_dK = ttest_1samp(dK_item, 0)

print(f"\n  Significance (cluster-corrected, n={len(unique_items_sig)} items):")
print(f"    Δ(S) ≠ 0: t={t_dS:.3f}, p={p_dS:.2e}")
print(f"    Δ(K) ≠ 0: t={t_dK:.3f}, p={p_dK:.2e}")

store.setdefault('binding', {})
store.setdefault('knowledge', {})
store['binding']['t_baseline'] = float(t_dS)
store['binding']['p_baseline'] = float(p_dS)
store['knowledge']['t_baseline'] = float(t_dK)
store['knowledge']['p_baseline'] = float(p_dK)


# Directional differentiation: P(R | a or b)
assoc_pos_arr = list(data['assoc_pos'])

# cond_p_R is imported from common.stats_utils (see setup cell).
pR_match = cond_p_R(_lps_s_base['B_cult'],  assoc_pos_arr)
pR_mism  = cond_p_R(_lps_s_base['B_unrel'], assoc_pos_arr)

pR_match_item = np.array([pR_match[items_arr_sig == it].mean() for it in unique_items_sig])
pR_mism_item  = np.array([pR_mism[items_arr_sig == it].mean()  for it in unique_items_sig])

t_dir, p_dir = ttest_1samp(pR_match_item, 0.5)

print(f"\n  Directional differentiation P(R | a or b):")
print(f"    Match    : mean = {pR_match_item.mean():.3f}  (t vs 0.5: t={t_dir:.2f}, p={p_dir:.2e})")
print(f"    Mismatch : mean = {pR_mism_item.mean():.3f}  [sanity ~0.5]")

store['binding']['pR_match_per_pair'] = pR_match.tolist()
store['binding']['pR_mism_per_pair']  = pR_mism.tolist()
store['binding']['pR_match_mean'] = float(pR_match_item.mean())
store['binding']['pR_mism_mean']  = float(pR_mism_item.mean())
store['binding']['t_directional'] = float(t_dir)
store['binding']['p_directional'] = float(p_dir)


store['knowledge']['logprobs_match']      = _lps_k_base['K_cult']
store['knowledge']['logprobs_mismatch']   = _lps_k_base['K_unrel']
store['binding']['logprobs_match']        = _lps_s_base['B_cult']
store['binding']['logprobs_mismatch']     = _lps_s_base['B_unrel']
store['binding']['logprobs_match_Rko']    = _lps_s_ko_B['B_cult']
store['binding']['logprobs_mismatch_Rko'] = _lps_s_ko_B['B_unrel']
store['binding']['logprobs_match_Uko']    = _lps_s_ko_A['B_cult']
store['binding']['logprobs_mismatch_Uko'] = _lps_s_ko_A['B_unrel']

_elapsed = _time.time() - _t_start
print(f"\n  Wall-clock time: {_elapsed:.1f} s  ({_elapsed/60:.2f} min)")
store['meta']['wall_clock_seconds'] = float(_elapsed)

save_results(store, suffix="_stage4_ko")

In [ ]:
# ================================================================
# VALIDATION GATE - reproduce published row + 8-array integrity
# ================================================================
import pickle
from pathlib import Path

from common.published_targets import TARGETS_BASE, ABS_TOL_BASE, PCT_TOL_BASE

TARGETS = TARGETS_BASE
ABS_TOL = ABS_TOL_BASE
PCT_TOL = PCT_TOL_BASE

_pkl = OUTPUT_DIR / f"results_{MODEL_KEY}_base_stage4_ko.pkl"
print(f"  Reloading: {_pkl}")
with open(_pkl, "rb") as f:
    _store = pickle.load(f)

T = TARGETS.get(MODEL_KEY)
assert T is not None, f"No TARGETS entry for {MODEL_KEY}"

_ok = True
def _check(cond, msg):
    global _ok
    mark = "OK  " if cond else "FAIL"
    print(f"  [{mark}] {msg}")
    if not cond:
        _ok = False

# 1. 8-array integrity
EXPECTED_KEYS = [
    ("knowledge", "logprobs_match"),
    ("knowledge", "logprobs_mismatch"),
    ("binding",   "logprobs_match"),
    ("binding",   "logprobs_mismatch"),
    ("binding",   "logprobs_match_Rko"),
    ("binding",   "logprobs_mismatch_Rko"),
    ("binding",   "logprobs_match_Uko"),
    ("binding",   "logprobs_mismatch_Uko"),
]
for sec, key in EXPECTED_KEYS:
    arr = _store.get(sec, {}).get(key)
    _check(arr is not None, f"{sec}.{key} present")
    if arr is not None:
        _check(len(arr) == 847, f"{sec}.{key} length == 847 (got {len(arr)})")
        sample = arr[0] if len(arr) else None
        _check(isinstance(sample, dict) and set(sample.keys()) == {"a", "b", "c"},
               f"{sec}.{key}[0] is dict with keys (a,b,c)")

_assoc = set(_store["factorial"]["assoc_pos"])
_check(_assoc.issubset({"a", "b"}), f"factorial.assoc_pos subset of (a,b)  (got {_assoc})")
_items_unique = len(set(_store["factorial"]["items"]))
_check(_items_unique == 66, f"unique factorial.items == 66 (got {_items_unique})")

# 2. Reproduction
b = _store["knockout"]["binding"]
k = _store["knockout"]["knowledge"]
abs_dS = abs(b["delta_baseline"])
abs_dK = abs(k["delta_baseline"])
red_B_S = b["reduction_B_pct"]
red_A_S = b["reduction_A_pct"]
red_B_K = k["reduction_B_pct"]
red_A_K = k["reduction_A_pct"]
KS_ratio = red_B_K / red_B_S if abs(red_B_S) > 1e-6 else float("nan")

def _within(val, target, tol, label):
    if target is None:
        print(f"  [SKIP] {label} (no published target)")
        return
    dev = abs(val - target)
    cond = dev < tol
    _check(cond, f"{label}: target={target:+.4f}  got={val:+.4f}  dev={dev:.4f}  tol={tol:.4f}")

_within(abs_dS, T["abs_dS"], ABS_TOL, "|dS| baseline")
_within(abs_dK, T["abs_dK"], ABS_TOL, "|dK| baseline")
_within(red_B_S, T["red_B_S"], PCT_TOL, "R->item dS reduction (pp)")
_within(red_B_K, T["red_B_K"], PCT_TOL, "R->item dK reduction (pp)")
_within(red_A_S, T["red_A_S"], PCT_TOL, "U->item dS reduction (pp)")
_within(red_A_K, T["red_A_K"], PCT_TOL, "U->item dK reduction (pp)")

print(f"\n  Reported metrics:")
print(f"    |dS| baseline      = {abs_dS:.4f}   (target {T['abs_dS']})")
print(f"    |dK| baseline      = {abs_dK:.4f}   (target {T['abs_dK']})")
print(f"    R->item dS red.    = {red_B_S:+.1f}%  (target {T['red_B_S']})")
print(f"    R->item dK red.    = {red_B_K:+.1f}%  (target {T['red_B_K']})")
print(f"    U->item dS red.    = {red_A_S:+.1f}%  (target {T['red_A_S']})  [control]")
print(f"    U->item dK red.    = {red_A_K:+.1f}%  (target {T['red_A_K']})  [control]")
print(f"    K/S ratio (R-KO)   = {KS_ratio:.2f}   (target {T['KS_ratio']})")
print(f"    wall-clock         = {_store['meta'].get('wall_clock_seconds', float('nan')):.1f} s")
print(f"    final path         = {_pkl}")

print(f"\n  === GATE [{MODEL_KEY}]: {'GREEN - faithful regen' if _ok else 'RED - investigate'} ===")
